In [8]:
import os
from pathlib import Path
from google.colab import drive

# --- CẤU HÌNH ĐƯỜNG DẪN GỐC ---
BASE_DIR = Path('/content/drive')

# --- KẾT NỐI GOOGLE DRIVE ---
if not BASE_DIR.exists():
    print("🔌 Đang kết nối Google Drive...")
    drive.mount(str(BASE_DIR))
else:
    print("✅ Google Drive đã được kết nối sẵn sàng.")

✅ Google Drive đã được kết nối sẵn sàng.


#Gán nhãn dữ liệu

In [11]:
import pandas as pd
import logging
import sys
from pathlib import Path

# --- 1. CẤU HÌNH LOGGING (FIX LỖI KHÔNG HIỆN) ---
# Xóa các cấu hình cũ để tránh xung đột trên Colab
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

# Cấu hình mới: Ép xuất ra màn hình (sys.stdout)
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    datefmt='%H:%M:%S',
    handlers=[logging.StreamHandler(sys.stdout)] # Quan trọng: Xuất ra màn hình console
)
logger = logging.getLogger()

# --- 2. CẤU HÌNH ĐƯỜNG DẪN & TÊN FILE ---
if 'BASE_DIR' not in globals():
    BASE_DIR = Path('/content/drive')

DATA_DIR = BASE_DIR / 'MyDrive/2026'

# Tên file đầu vào
INPUT_FILENAME = 'dataset.csv'
# Tên file đầu ra (Bạn đặt tên tại đây)
OUTPUT_FILENAME = 'step1_labeled_raw.csv'

INPUT_PATH = DATA_DIR / INPUT_FILENAME
OUTPUT_PATH = DATA_DIR / OUTPUT_FILENAME

# --- 3. HÀM XỬ LÝ CHÍNH ---
def process_labeling():
    print("\n" + "="*40)
    logger.info("🚀 BẮT ĐẦU CHẠY CODE...")

    # Kiểm tra file đầu vào
    if not INPUT_PATH.exists():
        logger.error(f"❌ LỖI: Không tìm thấy file tại {INPUT_PATH}")
        return

    try:
        # Đọc file
        logger.info(f"📂 Đang đọc file: {INPUT_FILENAME}")
        df = pd.read_csv(INPUT_PATH, low_memory=False)

        # --- LOGIC XỬ LÝ ---
        # 1. Chuyển score sang số
        df['score'] = pd.to_numeric(df['score'], errors='coerce')
        df = df.dropna(subset=['score'])
        df['score'] = df['score'].astype(int)

        # 2. Loại bỏ 3 sao
        before_count = len(df)
        df = df[df['score'] != 3].copy()
        logger.info(f"🧹 Đã lọc bỏ {before_count - len(df)} dòng 3 sao (Neutral).")

        # 3. Gán nhãn (0: <=2, 1: >=4)
        df['label'] = 0
        df.loc[df['score'] >= 4, 'label'] = 1

        # 4. Đổi tên cột
        if 'content' in df.columns:
            df.rename(columns={'content': 'text'}, inplace=True)

        final_df = df[['label', 'text']] if 'text' in df.columns else df

        # --- LƯU FILE ---
        final_df.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')

        print("-" * 40)
        logger.info(f"🎉 THÀNH CÔNG! File đã lưu tại:")
        logger.info(f"💾 {OUTPUT_PATH}")
        logger.info(f"📊 Tổng số dòng: {len(final_df):,}")

        # Thống kê
        stats = final_df['label'].value_counts().to_dict()
        logger.info(f"📈 Phân bố nhãn: {stats}")
        print("="*40 + "\n")

    except Exception as e:
        logger.error(f"🔥 LỖI FATAL: {e}")

# Chạy hàm
if __name__ == "__main__":
    process_labeling()


03:16:56 | INFO | 🚀 BẮT ĐẦU CHẠY CODE...
03:16:56 | INFO | 📂 Đang đọc file: dataset.csv
03:16:59 | INFO | 🧹 Đã lọc bỏ 8459 dòng 3 sao (Neutral).
----------------------------------------
03:17:01 | INFO | 🎉 THÀNH CÔNG! File đã lưu tại:
03:17:01 | INFO | 💾 /content/drive/MyDrive/2026/step1_labeled_raw.csv
03:17:01 | INFO | 📊 Tổng số dòng: 125,597
03:17:01 | INFO | 📈 Phân bố nhãn: {1: 95735, 0: 29862}



#Tiền xử lý

In [12]:
!pip install laonlp tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.3/19.3 MB 64.6 MB/s eta 0:00:00


In [13]:
import pandas as pd
import re
import unicodedata
import logging
import sys
from pathlib import Path
from tqdm.auto import tqdm
from laonlp.tokenize import word_tokenize

# Đăng ký tqdm với pandas để hiển thị thanh tiến trình
tqdm.pandas()

In [14]:

# --- 1. CẤU HÌNH LOGGING ---
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    datefmt='%H:%M:%S',
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger()

# --- 2. CẤU HÌNH ĐƯỜNG DẪN & FILE ---
if 'BASE_DIR' not in globals():
    BASE_DIR = Path('/content/drive')

DATA_DIR = BASE_DIR / 'MyDrive/2026'

# Input: Lấy từ kết quả của Bước 1
INPUT_FILENAME = 'step1_labeled_raw.csv'
# Output: Kết quả sạch cho Bước 2
OUTPUT_FILENAME = 'step2_cleaned_tokenized.csv'

INPUT_PATH = DATA_DIR / INPUT_FILENAME
OUTPUT_PATH = DATA_DIR / OUTPUT_FILENAME

# --- 3. CẤU HÌNH TỪ ĐIỂN & REGEX (COMPILE TRƯỚC ĐỂ TĂNG TỐC) ---
# Danh sách từ rác (Stopwords/Noise)
LAO_NOISE_WORDS = {
    "ເດີ", "ເດ", "ນໍ", "ເນາະ", "ຫວາ", "ວະ", "ນະ", "ນ່າ",
    "ຈ້າ", "ເອີ", "ອື", "ໂອເຄ", "ຊື່ໆ", "ຫັ້ນ", "ລະ", "ຕິ",
    "ເດີ້", "ແດ່", "ດອກ", "ແຫຼະ"
}

# Regex Patterns (Biên dịch sẵn)
# 1. Ký tự lặp (VD: ມາກກກ -> ມາກ)
RE_REPEAT_CHARS = re.compile(r'(.)\1+')
# 2. URL, Email
RE_URL_EMAIL = re.compile(r'http\S+|www\.\S+|\S+@\S+')
# 3. Ký tự ẩn unicode
RE_HIDDEN_CHARS = re.compile(r'[\u200b\u200e\u200f\ufeff]')
# 4. Giữ lại: Lào, Thái, Anh, Số, Dấu câu cơ bản. Loại bỏ icon/emoji lạ
# Phạm vi Unicode: \u0E80-\u0EFF (Lào), \u0E00-\u0E7F (Thái)
RE_ALLOW_CHARS = re.compile(r'[^\w\s\u0E80-\u0EFF\u0E00-\u0E7F\.,!\?]')
# 5. Khoảng trắng thừa
RE_WHITESPACE = re.compile(r'\s+')

# --- 4. HÀM XỬ LÝ TEXT ---
def clean_lao_text(text):
    """
    Hàm làm sạch và tách từ cho tiếng Lào.
    Input: Chuỗi thô
    Output: Chuỗi đã tách từ (cách nhau bởi khoảng trắng) hoặc None nếu rỗng
    """
    if not isinstance(text, str) or not text.strip():
        return None

    # 1. Chuẩn hóa Unicode (NFKC) & Chữ thường
    text = unicodedata.normalize('NFKC', text).lower()

    # 2. Xử lý Teencode (Ký tự lặp)
    # Lưu ý: r'\1' sẽ đưa 'ມາກກກ' thành 'ມາກ'.
    text = RE_REPEAT_CHARS.sub(r'\1', text)

    # 3. Xóa URL, Email, Ký tự ẩn
    text = RE_URL_EMAIL.sub('', text)
    text = RE_HIDDEN_CHARS.sub('', text)

    # 4. Lọc ký tự đặc biệt (Chỉ giữ Lào, Thái, Anh, Số)
    text = RE_ALLOW_CHARS.sub(' ', text)

    try:
        # 5. TÁCH TỪ (Word Segmentation)
        tokens = word_tokenize(text)

        # 6. LỌC TỪ RÁC
        final_tokens = [t for t in tokens if t not in LAO_NOISE_WORDS]

        # Nối lại
        text = " ".join(final_tokens)
    except Exception:
        # Trường hợp lỗi tách từ, trả về text đã clean cơ bản
        pass

    # 7. Xóa khoảng trắng thừa lần cuối
    text = RE_WHITESPACE.sub(' ', text).strip()

    return text if len(text) >= 2 else None

# --- 5. QUY TRÌNH CHẠY CHÍNH ---
def process_cleaning():
    print("\n" + "="*40)
    logger.info("🚀 BẮT ĐẦU CLEANING & TOKENIZING...")

    if not INPUT_PATH.exists():
        logger.error(f"❌ LỖI: Không tìm thấy file {INPUT_FILENAME}")
        logger.info("👉 Hãy chạy lại Bước 1 (Labeling) để tạo file này.")
        return

    try:
        # Đọc dữ liệu
        logger.info(f"📂 Đang đọc file: {INPUT_FILENAME}")
        df = pd.read_csv(INPUT_PATH)
        initial_len = len(df)

        # Kiểm tra cột text
        if 'text' not in df.columns:
            logger.error("❌ File input thiếu cột 'text'.")
            return

        # --- APPLY HÀM CLEANING (Có Progress Bar) ---
        logger.info("⚙️ Đang xử lý văn bản (quá trình này có thể mất chút thời gian)...")
        # progress_apply giúp hiện thanh chạy %
        df['text_clean'] = df['text'].progress_apply(clean_lao_text)

        # --- HẬU XỬ LÝ ---
        # 1. Bỏ dòng None (Do hàm clean trả về None nếu chuỗi rỗng/quá ngắn)
        df_clean = df.dropna(subset=['text_clean']).copy()

        # 2. Khử trùng lặp (Deduplication)
        # Loại bỏ các mẫu mà sau khi clean có nội dung y hệt nhau
        df_final = df_clean.drop_duplicates(subset=['text_clean', 'label'])

        # 3. Chọn cột cuối cùng để lưu
        # Đổi tên text_clean -> text để chuẩn hóa
        final_output = df_final[['label', 'text_clean']].rename(columns={'text_clean': 'text'})

        # --- LƯU FILE ---
        final_output.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')

        print("-" * 40)
        logger.info(f"🎉 HOÀN TẤT! Dữ liệu sạch đã sẵn sàng.")
        logger.info(f"💾 File lưu tại: {OUTPUT_FILENAME}")

        # Thống kê chi tiết
        removed_rows = initial_len - len(final_output)
        logger.info(f"📊 Tổng số mẫu ban đầu: {initial_len:,}")
        logger.info(f"🗑️ Đã loại bỏ: {removed_rows:,} mẫu (Rác/Trùng lặp/Quá ngắn)")
        logger.info(f"✅ Tổng số mẫu sạch: {len(final_output):,}")

        stats = final_output['label'].value_counts().to_dict()
        logger.info(f"📈 Phân bố nhãn Train: {stats}")

        print("\n📝 VÍ DỤ DỮ LIỆU ĐÃ TOKENIZE:")
        print(final_output.head())
        print("="*40 + "\n")

    except Exception as e:
        logger.error(f"🔥 LỖI FATAL: {e}")

if __name__ == "__main__":
    process_cleaning()


03:22:10 | INFO | 🚀 BẮT ĐẦU CLEANING & TOKENIZING...
03:22:10 | INFO | 📂 Đang đọc file: step1_labeled_raw.csv
03:22:11 | INFO | ⚙️ Đang xử lý văn bản (quá trình này có thể mất chút thời gian)...


  0%|          | 0/125597 [00:00<?, ?it/s]

----------------------------------------
03:22:22 | INFO | 🎉 HOÀN TẤT! Dữ liệu sạch đã sẵn sàng.
03:22:22 | INFO | 💾 File lưu tại: step2_cleaned_tokenized.csv
03:22:22 | INFO | 📊 Tổng số mẫu ban đầu: 125,597
03:22:22 | INFO | 🗑️ Đã loại bỏ: 89,788 mẫu (Rác/Trùng lặp/Quá ngắn)
03:22:22 | INFO | ✅ Tổng số mẫu sạch: 35,809
03:22:22 | INFO | 📈 Phân bố nhãn Train: {1: 22737, 0: 13072}

📝 VÍ DỤ DỮ LIỆU ĐÃ TOKENIZE:
   label                                             text
0      1                                               ດີ
1      1                                             ລາ ລ
2      1             ເປັນຫຍັງ ແຕະ ເຂົ້າ bcelone ບໍ່ ເຂົ້າ
5      1                                       ລັດ ສະ ຫນີ
6      1  ບໍ່ ແຈ້ງ ເຕືອນ ເວລາ ເງິນ ເຂົ້າ ແລະ ເວລາ ເງິ ນອກ



#Chia train 80 train, 20 val

In [15]:
import os
from pathlib import Path
from google.colab import drive

# --- KẾT NỐI DRIVE ---
BASE_DIR = Path('/content/drive')
if not BASE_DIR.exists():
    drive.mount(str(BASE_DIR))
else:
    print("✅ Drive đã kết nối.")

✅ Drive đã kết nối.


In [16]:
import pandas as pd
import logging
import sys
from pathlib import Path
from sklearn.model_selection import train_test_split

# --- 1. CẤU HÌNH LOGGING ---
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    datefmt='%H:%M:%S',
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger()

# --- 2. CẤU HÌNH ĐƯỜNG DẪN ---
if 'BASE_DIR' not in globals():
    BASE_DIR = Path('/content/drive')

DATA_DIR = BASE_DIR / 'MyDrive/2026'

# Input: Lấy từ kết quả Bước 2
INPUT_FILENAME = 'step2_cleaned_tokenized.csv'

# Output: Thư mục chứa file đã chia
OUTPUT_DIR = DATA_DIR / 'dataset_split'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True) # Tạo thư mục nếu chưa có

INPUT_PATH = DATA_DIR / INPUT_FILENAME
TRAIN_PATH = OUTPUT_DIR / 'train.csv'
VAL_PATH = OUTPUT_DIR / 'val.csv'

# --- 3. HÀM CHIA DỮ LIỆU ---
def split_dataset():
    print("\n" + "="*40)
    logger.info("🚀 BẮT ĐẦU CHIA TẬP DỮ LIỆU (SPLIT)...")

    # Kiểm tra file input
    if not INPUT_PATH.exists():
        logger.error(f"❌ LỖI: Không tìm thấy file {INPUT_FILENAME}")
        logger.info("👉 Hãy chạy lại Bước 2 (Cleaning) trước.")
        return

    try:
        # Đọc dữ liệu
        logger.info(f"📂 Đang đọc file: {INPUT_FILENAME}")
        df = pd.read_csv(INPUT_PATH)

        # Kiểm tra tính hợp lệ
        if 'label' not in df.columns:
            logger.error("❌ File thiếu cột 'label'. Không thể chia tầng (stratify).")
            return

        total_samples = len(df)
        logger.info(f"📊 Tổng số mẫu khả dụng: {total_samples:,}")

        # --- THỰC HIỆN CHIA (80/20) ---
        # stratify=df['label']: Giữ nguyên tỷ lệ Pos/Neg ở cả 2 tập
        logger.info("✂️ Đang chia dữ liệu (Train=80%, Val=20%)...")

        train_df, val_df = train_test_split(
            df,
            test_size=0.2,
            random_state=42,
            stratify=df['label'],
            shuffle=True
        )

        # --- LƯU FILE ---
        logger.info(f"💾 Đang lưu file vào thư mục: {OUTPUT_DIR.name}/")
        train_df.to_csv(TRAIN_PATH, index=False, encoding='utf-8-sig')
        val_df.to_csv(VAL_PATH, index=False, encoding='utf-8-sig')

        # --- BÁO CÁO KẾT QUẢ ---
        print("-" * 40)
        logger.info("✅ HOÀN TẤT QUÁ TRÌNH CHIA DỮ LIỆU.")

        # Hàm in thống kê đẹp
        def print_stats(name, data):
            count = len(data)
            pos = len(data[data['label'] == 1])
            neg = len(data[data['label'] == 0])
            pos_ratio = (pos/count)*100 if count > 0 else 0
            return f"{name:<10} | Tổng: {count:>6,} | Pos: {pos:>5,} ({pos_ratio:.1f}%) | Neg: {neg:>5,}"

        print("\n📊 THỐNG KÊ CHI TIẾT:")
        print(print_stats("TRAIN SET", train_df))
        print(print_stats("VAL SET", val_df))
        print(f"\n📂 Đường dẫn Train: {TRAIN_PATH}")
        print(f"📂 Đường dẫn Val:   {VAL_PATH}")
        print("="*40 + "\n")

    except Exception as e:
        logger.error(f"🔥 LỖI FATAL: {e}")

if __name__ == "__main__":
    split_dataset()


03:24:52 | INFO | 🚀 BẮT ĐẦU CHIA TẬP DỮ LIỆU (SPLIT)...
03:24:52 | INFO | 📂 Đang đọc file: step2_cleaned_tokenized.csv
03:24:52 | INFO | 📊 Tổng số mẫu khả dụng: 35,809
03:24:52 | INFO | ✂️ Đang chia dữ liệu (Train=80%, Val=20%)...
03:24:53 | INFO | 💾 Đang lưu file vào thư mục: dataset_split/
----------------------------------------
03:24:53 | INFO | ✅ HOÀN TẤT QUÁ TRÌNH CHIA DỮ LIỆU.

📊 THỐNG KÊ CHI TIẾT:
TRAIN SET  | Tổng: 28,647 | Pos: 18,189 (63.5%) | Neg: 10,458
VAL SET    | Tổng:  7,162 | Pos: 4,548 (63.5%) | Neg: 2,614

📂 Đường dẫn Train: /content/drive/MyDrive/2026/dataset_split/train.csv
📂 Đường dẫn Val:   /content/drive/MyDrive/2026/dataset_split/val.csv



# ==============================================================================
# CHIẾN THUẬT 1: FULL FINE-TUNING (ADVANCED VERSION)
# Tính năng: Class Weights (Cân bằng dữ liệu) + Early Stopping + Hardware Log
# ==============================================================================

In [18]:
#  1. CÀI ĐẶT THƯ VIỆN (Chạy 1 lần nếu chưa cài)
!pip install transformers datasets evaluate accelerate scikit-learn seaborn matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.8 MB/s eta 0:00:00


In [19]:
# ==============================================================================
# CHIẾN THUẬT 1: FULL FINE-TUNING (ADVANCED VERSION)
# Tính năng: Class Weights (Cân bằng dữ liệu) + Early Stopping + Hardware Log
# ==============================================================================

import pandas as pd
import numpy as np
import os
import torch
import time
import datetime
import psutil
import gc
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    TrainerCallback,
    EarlyStoppingCallback
)
import evaluate
import torch.nn as nn

# --- 1. CẤU HÌNH ---
MODEL_NAME = "w11wo/lao-roberta-base"  # Model bản địa (Chạy cái này trước)
# MODEL_NAME = "xlm-roberta-base"      # Chạy sau nếu cần so sánh

# Cấu hình Train khắt khe hơn (theo yêu cầu Research)
EPOCHS = 20              # Tăng lên 20 (nhưng sẽ dừng sớm nếu ko hiệu quả)
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
PATIENCE = 4             # Nếu 4 vòng liên tiếp ko tốt lên thì dừng
MAX_LEN = 128

# Đường dẫn (Giữ nguyên của anh)
BASE_DIR = '/content/drive/MyDrive/2026'
TRAIN_PATH = os.path.join(BASE_DIR, 'dataset_split/train.csv')
VAL_PATH = os.path.join(BASE_DIR, 'dataset_split/val.csv')
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
OUTPUT_DIR = os.path.join(BASE_DIR, 'Research_Models', f"Full_Tuning_{timestamp}")

os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- 2. HARDWARE CHECK & LOGGING ---
print("="*60)
print("🛠️ RESEARCH ENVIRONMENT SETUP")
print("="*60)
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    print(f"🔥 GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.memory_allocated(0)/1024**3:.2f}GB / {torch.cuda.get_device_properties(0).total_memory/1024**3:.2f}GB")
else:
    print("⚠️ WARNING: Running on CPU!")

# --- 3. XỬ LÝ DỮ LIỆU & TÍNH CLASS WEIGHTS (QUAN TRỌNG) ---
print("\n⚖️ ĐANG TÍNH TOÁN CÂN BẰNG DỮ LIỆU (CLASS WEIGHTS)...")
train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)

# Tính trọng số: Lớp nào ít dữ liệu thì trọng số cao hơn
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df['label']),
    y=train_df['label']
)
# Chuyển sang Tensor để nạp vào GPU
weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

print(f"   -> Tỉ lệ dữ liệu: Negative (0) vs Positive (1)")
print(f"   -> Trọng số áp dụng: Neg={class_weights[0]:.2f} | Pos={class_weights[1]:.2f}")
print("   (Model sẽ bị phạt nặng hơn gấp {:.1f} lần nếu đoán sai lớp Negative)".format(class_weights[0]/class_weights[1]))

# --- 4. PREPARE DATASET ---
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
def tokenize_func(examples):
    return tokenizer(examples["text"], truncation=True, max_length=MAX_LEN)

dataset = DatasetDict({
    'train': Dataset.from_pandas(train_df),
    'validation': Dataset.from_pandas(val_df)
})
tokenized_datasets = dataset.map(tokenize_func, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# --- 5. CUSTOM TRAINER (ĐỂ ÁP DỤNG TRỌNG SỐ) ---
# Đây là kỹ thuật nâng cao để chèn Class Weights vào Loss Function
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        # Sử dụng CrossEntropyLoss với trọng số đã tính
        loss_fct = nn.CrossEntropyLoss(weight=weights_tensor)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# --- 6. SETUP MODEL & METRICS ---
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(device)

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    acc = accuracy.compute(predictions=predictions, references=labels)
    f1_macro = f1.compute(predictions=predictions, references=labels, average="macro")
    return {"accuracy": acc["accuracy"], "f1_macro": f1_macro["f1"]}

# --- 7. TRAINING CONFIGURATION ---
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=32,
    num_train_epochs=EPOCHS,             # 20 Epochs
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",    # Tối ưu theo F1 thay vì Loss
    save_total_limit=2,                  # Chỉ giữ 2 checkpoint tốt nhất để tiết kiệm Drive
    fp16=True if device=="cuda" else False,
    report_to="none"
)

# Callback ghi log thời gian
class TimeLogCallback(TrainerCallback):
    def on_epoch_begin(self, args, state, control, **kwargs):
        self.start = time.time()
        print(f"\n⏱️  Epoch {state.epoch + 1} Start: {datetime.datetime.now().strftime('%H:%M:%S')}")
    def on_epoch_end(self, args, state, control, **kwargs):
        print(f"✅ Epoch {state.epoch} End - Duration: {time.time() - self.start:.2f}s")

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        TimeLogCallback(),
        EarlyStoppingCallback(early_stopping_patience=PATIENCE) # Tự động dừng nếu không tốt hơn
    ]
)

# --- 8. START TRAINING ---
print("\n" + "="*60)
print(f"🚀 BẮT ĐẦU HUẤN LUYỆN: {MODEL_NAME}")
print(f"🎯 Mục tiêu: Chạy tối đa {EPOCHS} epochs (Dừng sớm nếu {PATIENCE} epochs không tăng F1)")
print("="*60)

start_train = time.time()
trainer.train()
total_train_time = time.time() - start_train

# --- 9. LƯU KẾT QUẢ & CSV BÁO CÁO (FINAL STEP) ---
print("\n📊 ĐANG TẠO BÁO CÁO CHI TIẾT (VALIDATION REPORT)...")

# Dự đoán lại lần cuối với model tốt nhất
preds_output = trainer.predict(tokenized_datasets["validation"])
y_pred = np.argmax(preds_output.predictions, axis=1)
probs = torch.nn.functional.softmax(torch.tensor(preds_output.predictions), dim=-1).numpy()
confidence = np.max(probs, axis=1)

# Lưu CSV chi tiết
val_df['predicted_label'] = y_pred
val_df['confidence_score'] = confidence
val_df['is_correct'] = val_df['label'] == val_df['predicted_label']
csv_path = os.path.join(OUTPUT_DIR, 'validation_results_detailed.csv')
val_df.to_csv(csv_path, index=False)

# Lưu Model
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Ghi file log cứng
with open(os.path.join(OUTPUT_DIR, "training_info.txt"), "w") as f:
    f.write(f"Model: {MODEL_NAME}\n")
    f.write(f"Class Weights Used: Neg={class_weights[0]:.4f}, Pos={class_weights[1]:.4f}\n")
    f.write(f"Total Training Time: {total_train_time:.2f}s\n")
    f.write(f"Best F1-Macro: {trainer.state.best_metric:.4f}\n")

print("-" * 60)
print(f"🎉 HOÀN TẤT CHIẾN THUẬT 1!")
print(f"👉 Model lưu tại: {OUTPUT_DIR}")
print(f"👉 File CSV báo cáo: {csv_path}")
print("-" * 60)

🛠️ RESEARCH ENVIRONMENT SETUP
🔥 GPU: Tesla T4
   VRAM: 0.00GB / 14.56GB

⚖️ ĐANG TÍNH TOÁN CÂN BẰNG DỮ LIỆU (CLASS WEIGHTS)...
   -> Tỉ lệ dữ liệu: Negative (0) vs Positive (1)
   -> Trọng số áp dụng: Neg=1.37 | Pos=0.79
   (Model sẽ bị phạt nặng hơn gấp 1.7 lần nếu đoán sai lớp Negative)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


03:28:33 | INFO | HTTP Request: HEAD https://huggingface.co/w11wo/lao-roberta-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
03:28:34 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/w11wo/lao-roberta-base/d12847d430b1d2fa305fcffdf85ed11ad70e18aa/config.json "HTTP/1.1 200 OK"
03:28:34 | INFO | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/w11wo/lao-roberta-base/d12847d430b1d2fa305fcffdf85ed11ad70e18aa/config.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

03:28:34 | INFO | HTTP Request: HEAD https://huggingface.co/w11wo/lao-roberta-base/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


📊 ĐANG TẠO BÁO CÁO CHI TIẾT (VALIDATION REPORT)...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

------------------------------------------------------------
🎉 HOÀN TẤT CHIẾN THUẬT 1!
👉 Model lưu tại: /content/drive/MyDrive/2026/Research_Models/Full_Tuning_20260218_0328
👉 File CSV báo cáo: /content/drive/MyDrive/2026/Research_Models/Full_Tuning_20260218_0328/validation_results_detailed.csv
------------------------------------------------------------
